# Paper Tables (Datasets & Model Parameters)

This notebook generates the dataset summary and model/control parameter tables used in the paper.
It mirrors the logic in `scripts/paper_tables.py`, but is organized for Jupyter execution.

In [ ]:
from __future__ import annotations
import ast
import gzip
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

PROJECT_ROOT = Path.cwd().parents[1]
OPT_PATH = PROJECT_ROOT / 'src' / 'experiments' / 'run_il_cache_opt.py'
OUT_DIR = PROJECT_ROOT / 'results' / 'paper_tables'
OUT_DIR.mkdir(parents=True, exist_ok=True)


## Helpers

In [ ]:
def load_opt_constants() -> Dict[str, Any]:
    source = OPT_PATH.read_text(encoding='utf-8')
    tree = ast.parse(source)
    wanted = {
        'FEATURE_SETS','FEATURE_WINDOWS','POLLUTION_WINDOW_SLOTS',
        'WIKIPEDIA_SEPTEMBER_2007','WIKIPEDIA_OKTOBER_2007','WIKI2018',
        'IL_NUM_GAPS','IL_POP_TOP_PERCENT','IL_LABEL_TOPK_ROUNDING','IL_LABEL_TIE_BREAK',
        'IL_SIGMOID_A','IL_SIGMOID_B','IL_MAX_CLASSIFIERS','IL_MISSING_GAP_VALUE',
        'CACHE_SIZE_PERCENTAGES','ADMISSION_CAPACITY_ALPHA','DRIFT_ALPHA_MIN','DRIFT_ALPHA_MAX',
        'DRIFT_SENSITIVITY','DRIFT_EMA_ALPHA','DRIFT_STATS_ALPHA','DRIFT_NORM_EPS','DRIFT_Z_CLIP',
        'DRIFT_WEIGHT_JSD','DRIFT_WEIGHT_OVERLAP','DRIFT_GAIN','DRIFT_ALPHA_FLOOR_MULT',
        'SCORE_GATE_TOP_PERCENT','FILL_RATIO','FILL_RATE','PRESSURE_MISS_GAMMA','SCORE_SPREAD_Q',
        'SCORE_SPREAD_EMA_ALPHA','SCORE_SPREAD_EPS','SCORE_QUALITY_MIN','SCORE_QUALITY_MAX',
        'SCORE_QUALITY_MIN_BOOST','ADMISSION_PRECISION_TARGET','ADMISSION_PRECISION_SENSITIVITY',
        'CAPACITY_ALPHA_SCALE_MIN',
    }
    consts = {}
    for node in tree.body:
        if isinstance(node, ast.Assign) and len(node.targets) == 1:
            target = node.targets[0]
            if isinstance(target, ast.Name) and target.id in wanted:
                try:
                    consts[target.id] = ast.literal_eval(node.value)
                except Exception:
                    pass
    return consts

def dataset_map(consts: Dict[str, Any]) -> Dict[str, Dict[str, Any]]:
    return {
        'september2007': consts['WIKIPEDIA_SEPTEMBER_2007'],
        'oktober2007': consts['WIKIPEDIA_OKTOBER_2007'],
        'wiki2018': consts['WIKI2018'],
    }

def parse_raw_wikibench_line(line: str) -> Optional[str]:
    parts = line.split()
    if len(parts) < 4:
        return None
    return ' '.join(parts[2:-1])

def count_distinct(path: str, total_requests: int) -> Optional[int]:
    if not Path(path).exists():
        return None
    seen = set()
    count = 0
    try:
        with gzip.open(path, 'rt', encoding='utf-8', errors='ignore') as f:
            for line in f:
                if count >= total_requests:
                    break
                line = line.strip()
                if not line:
                    continue
                obj_id = None
                if path.endswith('wiki2018.gz'):
                    parts = line.split()
                    if len(parts) >= 2:
                        obj_id = parts[1]
                else:
                    obj_id = parse_raw_wikibench_line(line)
                if obj_id is None:
                    continue
                seen.add(obj_id)
                count += 1
    except Exception:
        return None
    return len(seen)

def feature_dim(num_gaps: int, feature_set: str) -> int:
    if feature_set == 'A0':
        return num_gaps
    if feature_set == 'A1':
        return num_gaps + 1
    if feature_set == 'A2':
        return num_gaps + 4
    if feature_set == 'A3':
        return num_gaps + 5
    raise ValueError(f'Unknown feature_set: {feature_set}')

def md_table(headers: List[str], rows: List[List[Any]]) -> str:
    lines = ['| ' + ' | '.join(headers) + ' |', '| ' + ' | '.join(['---']*len(headers)) + ' |']
    for row in rows:
        lines.append('| ' + ' | '.join(str(v) for v in row) + ' |')
    return '\n'.join(lines)


## Table 1 — Trace Summary

In [ ]:
consts = load_opt_constants()
datasets = ['september2007', 'wiki2018']
ds_map = dataset_map(consts)

headers = ['Trace','Requests','DistinctObjects']
rows = []
for name in datasets:
    cfg = ds_map[name]
    total_requests = int(cfg['num_total_requests'])
    # compute_distinct=True will scan the trace (can take time)
    compute_distinct = True
    distinct = count_distinct(cfg['path'], total_requests) if compute_distinct else '(not computed)'
    rows.append([cfg['name'], total_requests, distinct])

table1_md = md_table(headers, rows)
print(table1_md)
(OUT_DIR / 'table1_trace_summary.md').write_text(table1_md + '\n', encoding='utf-8')


## Table 2 — Model/Control Parameters (Paper Symbols)

In [ ]:
L = consts['IL_NUM_GAPS']
rho = consts['IL_POP_TOP_PERCENT']
windows = consts['FEATURE_WINDOWS']

headers = ['Symbol','Description','Value']
rows = [
    ['L','Number of gap/recency features', L],
    ['ρ','Top-K labeling ratio (K = floor(ρ·N_t))', rho],
    ['a','Learn++.NSE sigmoid slope', consts['IL_SIGMOID_A']],
    ['b','Learn++.NSE sigmoid shift', consts['IL_SIGMOID_B']],
    ['H_max','Max learners (prune oldest)', consts['IL_MAX_CLASSIFIERS']],
    ['g_miss','Missing-gap padding value', consts['IL_MISSING_GAP_VALUE']],
    ['w_short','Short frequency window (slots)', windows['short']],
    ['w_mid','Mid frequency window (slots)', windows['mid']],
    ['w_long','Long frequency window (slots)', windows['long']],
    ['w_cum','Cumulative frequency window', 'all past slots'],
    ['C (%)','Cache sizes as % of distinct objects', list(consts['CACHE_SIZE_PERCENTAGES'])],
    ['α_0','Base admission rate', consts['ADMISSION_CAPACITY_ALPHA']],
    ['α_min','Min admission rate', consts['DRIFT_ALPHA_MIN']],
    ['α_max','Max admission rate', consts['DRIFT_ALPHA_MAX']],
    ['β_d','Drift EMA (d̄_t)', consts['DRIFT_EMA_ALPHA']],
    ['β_stats','Drift mean/var EMA', consts['DRIFT_STATS_ALPHA']],
    ['w_JSD','Drift weight for JSD', consts['DRIFT_WEIGHT_JSD']],
    ['w_OV','Drift weight for overlap', consts['DRIFT_WEIGHT_OVERLAP']],
    ['g_d','Drift gain', consts['DRIFT_GAIN']],
    ['α_floor','Alpha floor multiplier', consts['DRIFT_ALPHA_FLOOR_MULT']],
    ['γ_miss','Miss-rate pressure coefficient', consts['PRESSURE_MISS_GAMMA']],
    ['q','Score-spread quantile', consts['SCORE_SPREAD_Q']],
    ['β_s','Score-spread EMA', consts['SCORE_SPREAD_EMA_ALPHA']],
    ['q_min','Min quality multiplier', consts['SCORE_QUALITY_MIN']],
    ['q_max','Max quality multiplier', consts['SCORE_QUALITY_MAX']],
    ['δ_q','Quality min-boost (capacity effect)', consts['SCORE_QUALITY_MIN_BOOST']],
    ['p_target','Admission precision target', consts['ADMISSION_PRECISION_TARGET']],
    ['κ_p','Precision sensitivity', consts['ADMISSION_PRECISION_SENSITIVITY']],
    ['κ_c,min','Min capacity scale', consts['CAPACITY_ALPHA_SCALE_MIN']],
    ['ϕ_gate','Score-gate top percentile', consts['SCORE_GATE_TOP_PERCENT']],
    ['ϕ_fill','Fill-phase ratio', consts['FILL_RATIO']],
    ['r_fill','Fill-phase minimum rate', consts['FILL_RATE']],
    ['W_poll','Pollution window (slots)', consts['POLLUTION_WINDOW_SLOTS']],
    ['Base learner','Classifier used in ensemble','GaussianNaiveBayes'],
    ['Feature set (gap-only)','x = [gap1..gapL], dim L (A0 ablation)', feature_dim(L,'A0')],
    ['Feature set (gap + multi-timescale freq)','x = [gap1..gapL, f_short, f_mid, f_long, f_cum], dim L+4 (A2 ablation)', feature_dim(L,'A2')],
]

table2_md = md_table(headers, rows)
print(table2_md)
(OUT_DIR / 'table2_model_params.md').write_text(table2_md + '\n', encoding='utf-8')


## (Optional) Ablation Figures

Add your plotting code here once you decide the exact ablation plots (e.g., HR vs cache size, Full vs NoDrift/NoGuardrails).